# 뉴스 데이터 xml 수집하기 - sbs news

In [1]:
import requests # 웹 서버에 요청을 보내고 웹 페이지의 데이터를 가져오는 라이브러리
from bs4 import BeautifulSoup # 가져온 데이터를 분석하고 원하는 부분만 추출하는 라이브러리

# 1. RSS 피드 주소로 접속해 데이터를 요청(하고, 그 결과를 news_rss 변수에 저장
news_rss = requests.get('https://news.sbs.co.kr/news/SectionRssFeed.do?sectionId=07')

# 2. 데이터 분석: 가져온 내용(news_rss.content)을 'xml' 형식으로 분석(파싱)하여 다루기 쉬운 형태(BeautifulSoup 객체)로 변환
news_rss_soup = BeautifulSoup(news_rss.content, 'xml')

# 3. 데이터 추출: 분석된 XML 데이터 안에서 <item> 태그 바로 아래에 있는 <link> 태그들을 모두 찾아 리스트 형태로 저장
link_list = news_rss_soup.select('item > link')

# 4. 결과 출력: 찾은 <link> 태그들이 총 몇 개인지 리스트의 길이(len)를 구해 출력
print("기사 개수:", len(link_list))

기사 개수: 29


In [2]:
# 1. 태그 찾기: XML 데이터에서 <item> 태그 바로 아래에 있는 <title> 태그들을 모두 찾기 
# 이때 title_list 안에는 텍스트가 아닌, 껍데기(<title>...</title>)까지 포함된 '태그 객체'들이 담기게 됨
title_list = news_rss_soup.select('item > title')

# 2. 텍스트만 추출: 태그 리스트를 하나씩 돌며 태그 껍데기를 벗기고 알맹이 글자(.text)만 뽑아낸 뒤 새 리스트를 만들어 title_list 변수에 덮어씌워 저장
title_list = [title.text for title in title_list]

# 3. 결과 출력: 순수한 기사 제목(텍스트)들만 담긴 최종 리스트를 화면에 출력
print(title_list)

["빗길·산악도 거뜬한 고성능 '강아지 로봇'…중국 유니트리 공개", '[바로이뉴스] "USA! USA!" 본격 \'미국 부흥회\'…"관세 위법" 대법원장 앞에선', "선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [취재파일]", '[속보] 트럼프, 집권 2기 첫 국정연설 시작', '군인 화장했더니 유골서 숟가락…"이게 뭐냐" 태국 발칵', '중국 드론 업체 DJI "증거 없이 수입 금지"…미국 법원에 소송', '미국, 인도·인도네시아 태양광 고율 관세…한화, 반사 이익?', '미국 최대 은행 JP모건도 AI발 대규모 인력 재배치 계획', '영국, 입국 전 전자여행허가 확대…한국 포함 85개국에 의무화', "'상호관세 돌려달라' 소송 봇물…로레알·다이슨도 합류", '멕시코 마약왕 사살 혼란에…혼다, 현지 공장 가동중단 뒤 재개', '술 먹는 침팬지…"야생 소변 샘플 20개 검사, 17개서 알코올 대사물 검출"', '미국, 유럽·중동에 군용기 150대 이동…이라크전 이후 최대', '미국, 서안지구서 첫 영사 서비스…이스라엘 "환영"', '이란, 미국과 핵협상 앞두고 "타결 가시권…외교 최우선해야"', '미국 법원, "오픈AI가 영업비밀 빼갔다" 주장한 xAI 소송 기각', '중국 누리꾼, "한국은 문화 도둑국" "중국설 훔쳤다" 주장', '길거리서 무료로…"참된 스승" 쏟아진 찬사', "스키 타다 '화들짝'…설원 가로질러 전력 질주", '노인 도왔더니 "4,500만 원 배상하라"…논란 터진 장면', '유엔총회, 우크라 지지 결의 채택…미·중은 기권', '오늘 국정연설…"백악관, 관세 15%로 인상 작업 중"', 'FBI국장이 왜 거기서 나와…미 하키팀 금메달 뒤풀이 참석 구설', '"미국, 은행에 고객 시민권정보 수집 요구 검토…이민단속 일환"', '보석 도둑맞은 루브르 박물관장 끝내 사임…마크롱 수락', '러 매체 "한국학자 란코프 국민대 교수, 라트비아서 체포"', '미국 유명앵커, 모친 실종 3주 만에 현상금 14억 내걸며 호소', '푸틴 "적들이 러

In [3]:
# 1. 뉴스 본문 수집
news_data = [] # 뉴스 본문 텍스트들을 모아둘 빈 리스트 준비

for link in link_list: # 만들어둔 링크 리스트에서 링크를 하나씩 꺼내어 반복
    # 태그 객체에서 순수 인터넷 주소(.text)만 뽑아내어 해당 기사 페이지로 접속 요청
    news_response = requests.get(link.text) 
    
    # 받아온 개별 뉴스 페이지의 HTML 데이터를 분석하기 쉽게 'html.parser'를 이용해 파싱
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser')
    
    # 기사 페이지 내에서 'itemprop 속성값이 articleBody인 div 태그'를 하나만 찾기 (실제 뉴스 본문)
    news_content = news_content_soup.select_one("div[itemprop=articleBody]")
    
    # 찾은 본문 태그에서 텍스트(알맹이)만 추출하여 처음 만든 빈 리스트(news_data)에 추가
    news_data.append(news_content.text) 


# 2. 데이터를 표로 만들고 파일로 저장
import pandas as pd # 데이터프레임을 만들고 파일로 저장하기 위해 데이터 분석 라이브러리 pandas를 불러옴

# '제목'과 '본문'을 짝지어 DataFrame 변환
news_df = pd.DataFrame(data={'title': title_list, 'content': news_data})
# 위 5개 행 미리보기
print(news_df.head())

# 완성된 표 데이터를 엑셀 호환 파일로 저장 (한글 깨짐을 막고, 불필요한 순번 열을 빼는 옵션)
news_df.to_csv("news.csv", encoding="utf-8-sig", index=False)

print("Save complete")

                                              title  \
0                빗길·산악도 거뜬한 고성능 '강아지 로봇'…중국 유니트리 공개   
1  [바로이뉴스] "USA! USA!" 본격 '미국 부흥회'…"관세 위법" 대법원장 앞에선   
2               선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [취재파일]   
3                         [속보] 트럼프, 집권 2기 첫 국정연설 시작   
4                    군인 화장했더니 유골서 숟가락…"이게 뭐냐" 태국 발칵   

                                             content  
0  \n\n▲ 유니트리 네 발 로봇 'As2'\n\n 중국 로봇 선두 주자인 유니트리(...  
1  \n  현지시간 24일 미 워싱턴 국회의사당 하원 회의장에서 트럼프 미 대통령의 집...  
2  \n\n관세 위법, 엡스타인 파동…진퇴양난 트럼프\n 도널드 트럼프 미 대통령 요즘...  
3  \n\n▲ 국정연설하는 트럼프 미 대통령\n\n 트럼프, 집권 2기 첫 국정연설 시...  
4  \n  태국에서 군 복무 중 숨진 군인의 유골에서 숟가락이 발견되는 일이 벌어져 태...  
Save complete


In [4]:
# print(news_response.url)
# print(news_response.status_code)
# print(news_content_soup.prettify()[:2000])  # 앞부분만 확인

# 뉴스 컨텐츠 클린징

In [5]:
news_data_1 = [] # 뉴스 본문 저장용 빈 리스트 생성

for link in link_list: # 수집한 링크 리스트 순회
    news_response = requests.get(link.text) # 개별 뉴스 페이지 접속 및 데이터 요청
    news_content_soup = BeautifulSoup(news_response.content, 'html.parser') # HTML 데이터 파싱

    news_content = news_content_soup.select_one("div[itemprop=articleBody]") # 본문 영역 태그 추출

    if news_content is not None: # 본문 태그가 정상적으로 존재하는 경우
          news_data_1.append(news_content.text.strip()) # 텍스트 추출 후 양끝 공백 제거하여 리스트에 추가
    else: # 본문 태그가 없는 경우 (에러 방지)
        news_data_1.append("본문 없음")  # "본문 없음" 텍스트로 대체하여 추가

# 제목 리스트와 본문 리스트를 결합하여 데이터프레임(표) 생성
news_data_1_df = pd.DataFrame(data={'title': title_list, 'content': news_data_1})

In [6]:
news_data_1_df.head()

,title,content
0,빗길·산악도 거뜬한 고성능 '강아지 로봇'…중국 유니트리 공개,▲ 유니트리 네 발 로봇 'As2'\n\n 중국 로봇 선두 주자인 유니트리(宇樹科技...
1,"[바로이뉴스] ""USA! USA!"" 본격 '미국 부흥회'…""관세 위법"" 대법원장 앞에선",현지시간 24일 미 워싱턴 국회의사당 하원 회의장에서 트럼프 미 대통령의 집권 2기...
2,선거 앞두고 줄줄이 악재…'최후통첩' 전쟁 결단하나 [취재파일],"관세 위법, 엡스타인 파동…진퇴양난 트럼프\n 도널드 트럼프 미 대통령 요즘 고민이..."
3,"[속보] 트럼프, 집권 2기 첫 국정연설 시작","▲ 국정연설하는 트럼프 미 대통령\n\n 트럼프, 집권 2기 첫 국정연설 시작 \n..."
4,"군인 화장했더니 유골서 숟가락…""이게 뭐냐"" 태국 발칵",태국에서 군 복무 중 숨진 군인의 유골에서 숟가락이 발견되는 일이 벌어져 태국 군 ...


In [7]:
missing_count = sum(1 for content in news_data_1 if content == "본문 없음")
print(f"본문이 없는 기사 개수: {missing_count}개")
print(f"전체 기사 개수: {len(news_data_1)}개")

본문이 없는 기사 개수: 0개
전체 기사 개수: 29개


In [8]:
import pandas as pd # pandas 라이브러리 호출

# 제목 리스트와 본문 리스트를 합쳐 데이터프레임(표) 생성
news_df = pd.DataFrame({
    'title': title_list,
    'content': news_data})

# 본문이 "본문 없음"과 같은지 확인하여 그 결과(True/False)를 'is_missing'이라는 새 열에 저장
news_df['is_missing'] = news_df['content'] == "본문 없음"

# 'is_missing' 열의 True 값들을 모두 더해 누락된 본문의 총 개수 계산
missing_count = news_df['is_missing'].sum()

# 데이터프레임의 전체 행(기사) 개수 계산
total_count = len(news_df)

# 누락된 개수, 전체 개수, 그리고 누락 비율(소수점 첫째 자리 퍼센트) 출력
print(f"본문 없음: {missing_count}개 / 전체: {total_count}개 ({missing_count/total_count:.1%})")

본문 없음: 0개 / 전체: 29개 (0.0%)
